# Unit 7 Practical: Complete System Simulation (Teacher)
## Cart-Pendulum with Motor Drive

### Objectives
- Simulate a complete multi-body dynamic system
- Compare numerical integration methods
- Implement external forcing and control
- Validate results through energy analysis
- Produce publication-quality visualizations

### System Description

A motor-driven cart on a horizontal track with an attached pendulum:

```
                   ●  m_p (pendulum mass)
                  /|
                /  |
              /    | L (pendulum length)
            /      |
          / θ      |
  ━━━━━━━━━━━━━━━━
  |   Cart m_c   |  ←── F_motor(t)
  ━━━━━━━━━━━━━━━━
  ═════════════════  (frictionless track)
```

**Key Features**:
- **Cart**: Mass $m_c$, position $x$, driven by motor force $F_{motor}(t)$
- **Pendulum**: Mass $m_p$, length $L$, angle $\theta$ from vertical
- **Forcing**: Time-varying motor force
- **Constraints**: Cart moves only horizontally, pendulum pivots at cart center

**Applications**:
- Inverted pendulum control
- Crane anti-sway systems
- Robot balance control
- Seismic vibration testing

### Learning Outcomes
1. Formulate multi-body EOM using Lagrangian mechanics
2. Implement system in simulation-ready form
3. Compare ODE solver performance
4. Analyze dynamic response
5. Validate through energy conservation

**Duration**: ~150 minutes (2.5 hours)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import time
from matplotlib.patches import Rectangle, Circle, FancyArrow
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Configure plotting
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

print("="*70)
print("UNIT 7 PRACTICAL: Cart-Pendulum System Simulation")
print("="*70)

## Part 1: System Formulation

### Lagrangian Approach

#### Generalized Coordinates
- $q_1 = x$ (cart position)
- $q_2 = \theta$ (pendulum angle from vertical)

#### Kinetic Energy
**Cart**: $T_c = \frac{1}{2}m_c\dot{x}^2$

**Pendulum** (position: $(x + L\sin\theta, L\cos\theta)$):
$$T_p = \frac{1}{2}m_p\left[(\dot{x} + L\dot{\theta}\cos\theta)^2 + (L\dot{\theta}\sin\theta)^2\right]$$

**Total**:
$$T = \frac{1}{2}(m_c + m_p)\dot{x}^2 + \frac{1}{2}m_pL^2\dot{\theta}^2 + m_pL\dot{x}\dot{\theta}\cos\theta$$

#### Potential Energy
$$V = -m_pgL\cos\theta$$ (taking cart level as zero)

#### Lagrangian
$$L = T - V$$

#### Equations of Motion
Using Euler-Lagrange equations with generalized force $F_{motor}$ on cart:

$$(m_c + m_p)\ddot{x} + m_pL\ddot{\theta}\cos\theta - m_pL\dot{\theta}^2\sin\theta = F_{motor}$$

$$m_pL^2\ddot{\theta} + m_pL\ddot{x}\cos\theta - m_pgL\sin\theta = 0$$

#### Convert to First-Order System
State vector: $\mathbf{y} = [x, \theta, \dot{x}, \dot{\theta}]^T$

$$\frac{d\mathbf{y}}{dt} = \begin{bmatrix} \dot{x} \\ \dot{\theta} \\ \ddot{x} \\ \ddot{\theta} \end{bmatrix}$$

Solve coupled equations for $\ddot{x}$ and $\ddot{\theta}$:

$$\ddot{x} = \frac{F_{motor} + m_pL\dot{\theta}^2\sin\theta + m_pg\sin\theta\cos\theta}{m_c + m_p\sin^2\theta}$$

$$\ddot{\theta} = \frac{-F_{motor}\cos\theta - m_pL\dot{\theta}^2\sin\theta\cos\theta + (m_c+m_p)g\sin\theta}{L(m_c + m_p\sin^2\theta)}$$

In [ ]:
# Part 1: System Parameters and Implementation
print("\n" + "="*70)
print("PART 1: System Formulation and Implementation")
print("="*70)

# Physical parameters
m_c = 2.0    # kg (cart mass)
m_p = 0.5    # kg (pendulum mass)
L = 1.0      # m (pendulum length)
g = 9.81     # m/s² (gravity)

print(f"\nSystem parameters:")
print(f"  Cart mass: m_c = {m_c} kg")
print(f"  Pendulum mass: m_p = {m_p} kg")
print(f"  Pendulum length: L = {L} m")
print(f"  Gravity: g = {g} m/s²")

# Motor forcing function - sinusoidal with envelope
def motor_force(t):
    """
    Time-varying motor force with:
    - Sinusoidal component at frequency f
    - Gaussian envelope centered at t_center
    """
    f = 0.5  # Hz
    t_center = 5.0  # s
    sigma = 2.0  # s
    amplitude = 5.0  # N
    
    envelope = np.exp(-(t - t_center)**2 / (2*sigma**2))
    return amplitude * envelope * np.sin(2*np.pi*f*t)

# System equations of motion
def cart_pendulum_ode(t, y):
    """
    Cart-pendulum equations of motion
    State: y = [x, θ, ẋ, θ̇]
    Returns: dy/dt = [ẋ, θ̇, ẍ, θ̈]
    """
    x, theta, x_dot, theta_dot = y
    
    # Motor force at current time
    F = motor_force(t)
    
    # Common denominator
    denom = m_c + m_p * np.sin(theta)**2
    
    # Accelerations from coupled equations
    x_ddot = (F + m_p*L*theta_dot**2*np.sin(theta) 
              + m_p*g*np.sin(theta)*np.cos(theta)) / denom
    
    theta_ddot = (-F*np.cos(theta) - m_p*L*theta_dot**2*np.sin(theta)*np.cos(theta)
                  + (m_c + m_p)*g*np.sin(theta)) / (L * denom)
    
    return [x_dot, theta_dot, x_ddot, theta_ddot]

# Initial conditions
x0 = 0.0           # Cart starts at origin
theta0 = np.pi/6   # Pendulum at 30° from vertical
x_dot0 = 0.0       # Cart at rest
theta_dot0 = 0.0   # Pendulum at rest

y0 = [x0, theta0, x_dot0, theta_dot0]

print(f"\nInitial conditions:")
print(f"  Cart position: x₀ = {x0} m")
print(f"  Pendulum angle: θ₀ = {np.degrees(theta0):.1f}° from vertical")
print(f"  Cart velocity: ẋ₀ = {x_dot0} m/s")
print(f"  Pendulum angular velocity: θ̇₀ = {theta_dot0} rad/s")

# Test motor forcing
t_test = np.linspace(0, 10, 500)
F_test = [motor_force(t) for t in t_test]

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(t_test, F_test, 'b-', linewidth=2)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Motor Force (N)')
ax.set_title('Motor Forcing Function: F(t) = A·exp(-(t-5)²/8)·sin(2πft)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figs/u7_p1_forcing.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ System formulation complete")
print(f"✓ Motor forcing function defined")
print("="*70)

## Part 2: Numerical Method Comparison

We'll compare four integration methods:

| Method | Order | Characteristics |
|--------|-------|-----------------|
| **RK4 (manual)** | 4 | Classical implementation |
| **RK23** | 3 | scipy adaptive, lower accuracy |
| **RK45** | 5 | scipy adaptive, general purpose |
| **DOP853** | 8 | scipy adaptive, high accuracy |

**Evaluation Criteria**:
1. **Accuracy**: Compare with high-precision reference
2. **Computational cost**: Function evaluations
3. **Energy conservation**: Physical validation
4. **Execution time**: Wall-clock performance

In [ ]:
# Part 2: Method Comparison
print("\n" + "="*70)
print("PART 2: Numerical Method Comparison")
print("="*70)

# Time span for simulation
t_span = (0, 10)
t_eval = np.linspace(0, 10, 500)

# RK4 manual implementation
def rk4_step(f, t, y, h):
    """Single RK4 step"""
    k1 = np.array(f(t, y))
    k2 = np.array(f(t + h/2, y + h*k1/2))
    k3 = np.array(f(t + h/2, y + h*k2/2))
    k4 = np.array(f(t + h, y + h*k3))
    return y + h*(k1 + 2*k2 + 2*k3 + k4)/6

def rk4_integrate(f, t_span, y0, n_steps):
    """Full RK4 integration"""
    t0, tf = t_span
    t = np.linspace(t0, tf, n_steps+1)
    h = (tf - t0) / n_steps
    
    y = np.zeros((n_steps+1, len(y0)))
    y[0] = y0
    
    for i in range(n_steps):
        y[i+1] = rk4_step(f, t[i], y[i], h)
    
    return t, y

# Compare methods
print(f"\nSimulating with different methods...")
print(f"Time span: {t_span[0]} to {t_span[1]} s")

methods = {
    'RK4 (h=0.01)': None,
    'RK23': 'RK23',
    'RK45': 'RK45',
    'DOP853': 'DOP853'
}

solutions = {}
timings = {}
function_evals = {}

# RK4 manual
start = time.time()
n_steps_rk4 = 1000  # h = 0.01
t_rk4, y_rk4 = rk4_integrate(cart_pendulum_ode, t_span, y0, n_steps_rk4)
timings['RK4 (h=0.01)'] = time.time() - start
function_evals['RK4 (h=0.01)'] = 4 * n_steps_rk4
solutions['RK4 (h=0.01)'] = (t_rk4, y_rk4.T)

print(f"\n  RK4 (manual, h=0.01):")
print(f"    Time: {timings['RK4 (h=0.01)']*1000:.2f} ms")
print(f"    Function evals: {function_evals['RK4 (h=0.01)']}")

# scipy methods
for name, method in methods.items():
    if method is None:
        continue
    
    start = time.time()
    sol = solve_ivp(cart_pendulum_ode, t_span, y0, method=method,
                    t_eval=t_eval, rtol=1e-8, atol=1e-11)
    timings[name] = time.time() - start
    function_evals[name] = sol.nfev
    solutions[name] = (sol.t, sol.y)
    
    print(f"\n  {name}:")
    print(f"    Time: {timings[name]*1000:.2f} ms")
    print(f"    Function evals: {sol.nfev}")
    print(f"    Success: {sol.success}")

# Energy calculation
def compute_energy(t, y):
    """Calculate total mechanical energy"""
    x, theta, x_dot, theta_dot = y
    
    # Kinetic energy
    T_cart = 0.5 * m_c * x_dot**2
    
    # Pendulum position and velocity
    x_p = x + L*np.sin(theta)
    y_p = L*np.cos(theta)
    vx_p = x_dot + L*theta_dot*np.cos(theta)
    vy_p = -L*theta_dot*np.sin(theta)
    
    T_pend = 0.5 * m_p * (vx_p**2 + vy_p**2)
    
    # Potential energy
    V = -m_p*g*y_p
    
    return T_cart + T_pend + V

# Calculate energies for all methods
energies = {}
for name, (t, y) in solutions.items():
    E = np.array([compute_energy(t[i], y[:, i]) for i in range(len(t))])
    energies[name] = E

# Work done by motor force
def work_done_by_motor(t, x):
    """Calculate cumulative work by motor force"""
    W = np.zeros_like(t)
    for i in range(1, len(t)):
        dt = t[i] - t[i-1]
        F = motor_force(t[i])
        dx = x[i] - x[i-1]
        W[i] = W[i-1] + F * dx
    return W

# Use DOP853 as reference (highest accuracy)
ref_name = 'DOP853'
ref_t, ref_y = solutions[ref_name]

# Visualization
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

colors = {'RK4 (h=0.01)': 'red', 'RK23': 'orange', 'RK45': 'blue', 'DOP853': 'green'}

# Cart position
ax1 = fig.add_subplot(gs[0, 0])
for name in ['RK4 (h=0.01)', 'RK23', 'RK45', 'DOP853']:
    t, y = solutions[name]
    ax1.plot(t, y[0], color=colors[name], linewidth=2, label=name, alpha=0.7)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Cart Position (m)')
ax1.set_title('Cart Position x(t)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Pendulum angle
ax2 = fig.add_subplot(gs[0, 1])
for name in ['RK4 (h=0.01)', 'RK23', 'RK45', 'DOP853']:
    t, y = solutions[name]
    ax2.plot(t, np.degrees(y[1]), color=colors[name], linewidth=2, label=name, alpha=0.7)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Angle (degrees)')
ax2.set_title('Pendulum Angle θ(t)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Phase space - cart
ax3 = fig.add_subplot(gs[0, 2])
for name in ['RK4 (h=0.01)', 'RK23', 'RK45', 'DOP853']:
    t, y = solutions[name]
    ax3.plot(y[0], y[2], color=colors[name], linewidth=1.5, label=name, alpha=0.7)
ax3.set_xlabel('Cart Position (m)')
ax3.set_ylabel('Cart Velocity (m/s)')
ax3.set_title('Cart Phase Portrait')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# Energy evolution
ax4 = fig.add_subplot(gs[1, 0])
for name in ['RK4 (h=0.01)', 'RK23', 'RK45', 'DOP853']:
    t, y = solutions[name]
    E = energies[name]
    ax4.plot(t, E, color=colors[name], linewidth=2, label=name, alpha=0.7)
    
# Add work done by motor
W_motor = work_done_by_motor(ref_t, ref_y[0])
ax4.plot(ref_t, energies[ref_name][0] + W_motor, 'k--', 
         linewidth=2, label='E₀ + W_motor', alpha=0.5)

ax4.set_xlabel('Time (s)')
ax4.set_ylabel('Energy (J)')
ax4.set_title('Total Mechanical Energy')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# Energy conservation check
ax5 = fig.add_subplot(gs[1, 1])
for name in ['RK4 (h=0.01)', 'RK23', 'RK45', 'DOP853']:
    t, y = solutions[name]
    E = energies[name]
    # Energy change should equal work done
    W = work_done_by_motor(t, y[0])
    energy_error = (E - (E[0] + W)) / np.abs(E[0]) * 100
    ax5.semilogy(t, np.abs(energy_error) + 1e-12, color=colors[name], 
                 linewidth=2, label=name, alpha=0.7)

ax5.set_xlabel('Time (s)')
ax5.set_ylabel('Energy Error (%)')
ax5.set_title('Energy Conservation: |E - (E₀ + W)| / E₀')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3, which='both')

# Method accuracy (vs DOP853)
ax6 = fig.add_subplot(gs[1, 2])
for name in ['RK4 (h=0.01)', 'RK23', 'RK45']:
    t, y = solutions[name]
    # Interpolate reference to same times
    x_ref = np.interp(t, ref_t, ref_y[0])
    theta_ref = np.interp(t, ref_t, ref_y[1])
    
    error_x = np.abs(y[0] - x_ref)
    error_theta = np.abs(y[1] - theta_ref)
    total_error = np.sqrt(error_x**2 + error_theta**2)
    
    ax6.semilogy(t, total_error + 1e-12, color=colors[name], 
                 linewidth=2, label=name, alpha=0.7)

ax6.set_xlabel('Time (s)')
ax6.set_ylabel('Position Error vs DOP853 (m, rad)')
ax6.set_title('Accuracy Comparison')
ax6.legend(fontsize=8)
ax6.grid(True, alpha=0.3, which='both')

# Computational efficiency
ax7 = fig.add_subplot(gs[2, 0])
names = list(methods.keys())
evals = [function_evals[name] for name in names]
times = [timings[name]*1000 for name in names]

x_pos = np.arange(len(names))
bars1 = ax7.bar(x_pos - 0.2, evals, 0.4, label='Function Evals', 
                color=[colors[n] for n in names], alpha=0.7)
ax7_twin = ax7.twinx()
bars2 = ax7_twin.bar(x_pos + 0.2, times, 0.4, label='Time (ms)', 
                     color=[colors[n] for n in names], alpha=0.4)

ax7.set_ylabel('Function Evaluations')
ax7_twin.set_ylabel('Execution Time (ms)')
ax7.set_xlabel('Method')
ax7.set_title('Computational Cost')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax7.legend(handles=[bars1], loc='upper left', fontsize=8)
ax7_twin.legend(handles=[bars2], loc='upper right', fontsize=8)
ax7.grid(True, alpha=0.3, axis='y')

# Efficiency: accuracy vs cost
ax8 = fig.add_subplot(gs[2, 1])
for name in ['RK4 (h=0.01)', 'RK23', 'RK45']:
    t, y = solutions[name]
    x_ref = np.interp(t, ref_t, ref_y[0])
    theta_ref = np.interp(t, ref_t, ref_y[1])
    max_error = np.max(np.sqrt((y[0] - x_ref)**2 + (y[1] - theta_ref)**2))
    
    ax8.loglog([function_evals[name]], [max_error], 'o', 
               color=colors[name], markersize=12, label=name)

ax8.set_xlabel('Function Evaluations')
ax8.set_ylabel('Max Error (m, rad)')
ax8.set_title('Efficiency: Accuracy vs Cost')
ax8.legend(fontsize=8)
ax8.grid(True, alpha=0.3, which='both')

# Summary statistics table
ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')

summary_text = "METHOD COMPARISON\n" + "="*40 + "\n\n"
for name in methods.keys():
    t, y = solutions[name]
    if name != 'DOP853':
        x_ref = np.interp(t, ref_t, ref_y[0])
        theta_ref = np.interp(t, ref_t, ref_y[1])
        max_err = np.max(np.sqrt((y[0]-x_ref)**2 + (y[1]-theta_ref)**2))
        summary_text += f"{name}:\n"
        summary_text += f"  F-evals: {function_evals[name]}\n"
        summary_text += f"  Time: {timings[name]*1000:.1f} ms\n"
        summary_text += f"  Max error: {max_err:.2e}\n\n"
    else:
        summary_text += f"{name} (reference):\n"
        summary_text += f"  F-evals: {function_evals[name]}\n"
        summary_text += f"  Time: {timings[name]*1000:.1f} ms\n\n"

summary_text += "="*40 + "\n"
summary_text += "CONCLUSION:\n"
summary_text += "• RK45: Best balance\n"
summary_text += "  of speed & accuracy\n"
summary_text += "• DOP853: Highest\n"
summary_text += "  accuracy, more cost\n"
summary_text += "• RK4: Predictable,\n"
summary_text += "  manual control\n"

ax9.text(0.05, 0.95, summary_text, transform=ax9.transAxes,
         fontsize=9, verticalalignment='top', family='monospace')

plt.savefig('figs/u7_p1_method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*70}")
print("Method Comparison Complete")
print("="*70)

## Part 3: System Visualization

Create a comprehensive visualization showing:
1. **System state**: Cart and pendulum configuration
2. **Trajectory**: Pendulum mass path
3. **Time histories**: Positions and velocities
4. **Energy flow**: Tracking energy transfer

This provides physical insight into system behavior.

In [ ]:
# Part 3: Comprehensive Visualization
print("\n" + "="*70)
print("PART 3: System Visualization and Analysis")
print("="*70)

# Use RK45 solution for visualization (good balance)
t_vis, y_vis = solutions['RK45']
x_vis = y_vis[0]
theta_vis = y_vis[1]
x_dot_vis = y_vis[2]
theta_dot_vis = y_vis[3]

# Calculate pendulum mass positions
x_pend = x_vis + L*np.sin(theta_vis)
y_pend = L*np.cos(theta_vis)

# Create comprehensive figure
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# 1. System animation frames
ax1 = fig.add_subplot(gs[0, :])
n_frames = 10
for i, idx in enumerate(np.linspace(0, len(t_vis)-1, n_frames, dtype=int)):
    alpha = 0.2 + 0.8 * (i/(n_frames-1))
    color = plt.cm.viridis(i/(n_frames-1))
    
    # Draw cart
    cart_width = 0.3
    cart_height = 0.15
    cart_x = x_vis[idx] - cart_width/2
    cart_y = 0
    rect = Rectangle((cart_x, cart_y), cart_width, cart_height,
                     linewidth=2, edgecolor=color, facecolor=color, alpha=alpha)
    ax1.add_patch(rect)
    
    # Draw pendulum
    ax1.plot([x_vis[idx], x_pend[idx]], [cart_height/2, y_pend[idx]], 
             'o-', color=color, linewidth=3, markersize=8, alpha=alpha)

# Draw pendulum trajectory
ax1.plot(x_pend, y_pend, 'r--', linewidth=1, alpha=0.5, label='Pendulum path')

# Draw track
track_extent = max(np.max(np.abs(x_vis)) + 1, 2)
ax1.plot([-track_extent, track_extent], [0, 0], 'k-', linewidth=4)

ax1.set_xlim(-track_extent, track_extent)
ax1.set_ylim(-0.3, 1.5)
ax1.set_xlabel('x (m)')
ax1.set_ylabel('y (m)')
ax1.set_title('Cart-Pendulum System Evolution (10 snapshots)')
ax1.set_aspect('equal')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Cart position and velocity
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(t_vis, x_vis, 'b-', linewidth=2, label='Position x')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Cart Position (m)', color='b')
ax2.tick_params(axis='y', labelcolor='b')
ax2.grid(True, alpha=0.3)

ax2_twin = ax2.twinx()
ax2_twin.plot(t_vis, x_dot_vis, 'r-', linewidth=2, label='Velocity ẋ')
ax2_twin.set_ylabel('Cart Velocity (m/s)', color='r')
ax2_twin.tick_params(axis='y', labelcolor='r')
ax2.set_title('Cart Motion')

# 3. Pendulum angle and angular velocity
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(t_vis, np.degrees(theta_vis), 'b-', linewidth=2, label='Angle θ')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Angle (degrees)', color='b')
ax3.tick_params(axis='y', labelcolor='b')
ax3.grid(True, alpha=0.3)

ax3_twin = ax3.twinx()
ax3_twin.plot(t_vis, theta_dot_vis, 'r-', linewidth=2, label='Angular vel. θ̇')
ax3_twin.set_ylabel('Angular Velocity (rad/s)', color='r')
ax3_twin.tick_params(axis='y', labelcolor='r')
ax3.set_title('Pendulum Motion')

# 4. Energy breakdown
ax4 = fig.add_subplot(gs[1, 2])
E_total = energies['RK45']
E0 = E_total[0]

# Breakdown energies
KE_cart = 0.5 * m_c * x_dot_vis**2
vx_p = x_dot_vis + L*theta_dot_vis*np.cos(theta_vis)
vy_p = -L*theta_dot_vis*np.sin(theta_vis)
KE_pend = 0.5 * m_p * (vx_p**2 + vy_p**2)
PE = -m_p*g*y_pend

ax4.fill_between(t_vis, 0, KE_cart, alpha=0.5, label='KE Cart', color='blue')
ax4.fill_between(t_vis, KE_cart, KE_cart+KE_pend, alpha=0.5, label='KE Pendulum', color='green')
ax4.fill_between(t_vis, KE_cart+KE_pend, E_total, alpha=0.5, label='PE', color='red')
ax4.plot(t_vis, E_total, 'k-', linewidth=2, label='Total')

ax4.set_xlabel('Time (s)')
ax4.set_ylabel('Energy (J)')
ax4.set_title('Energy Breakdown')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# 5. Phase portrait - cart
ax5 = fig.add_subplot(gs[2, 0])
scatter = ax5.scatter(x_vis, x_dot_vis, c=t_vis, cmap='viridis', 
                     s=10, alpha=0.6)
ax5.set_xlabel('Cart Position x (m)')
ax5.set_ylabel('Cart Velocity ẋ (m/s)')
ax5.set_title('Cart Phase Portrait')
ax5.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax5, label='Time (s)')

# 6. Phase portrait - pendulum
ax6 = fig.add_subplot(gs[2, 1])
scatter = ax6.scatter(np.degrees(theta_vis), theta_dot_vis, c=t_vis, 
                     cmap='viridis', s=10, alpha=0.6)
ax6.set_xlabel('Pendulum Angle θ (degrees)')
ax6.set_ylabel('Angular Velocity θ̇ (rad/s)')
ax6.set_title('Pendulum Phase Portrait')
ax6.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax6, label='Time (s)')

# 7. Pendulum trajectory in space
ax7 = fig.add_subplot(gs[2, 2])
scatter = ax7.scatter(x_pend, y_pend, c=t_vis, cmap='viridis', 
                     s=10, alpha=0.6)
ax7.plot(x_pend[0], y_pend[0], 'go', markersize=10, label='Start')
ax7.plot(x_pend[-1], y_pend[-1], 'ro', markersize=10, label='End')
ax7.set_xlabel('x (m)')
ax7.set_ylabel('y (m)')
ax7.set_title('Pendulum Mass Trajectory')
ax7.set_aspect('equal')
ax7.legend()
ax7.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax7, label='Time (s)')

plt.savefig('figs/u7_p1_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Comprehensive visualization complete")
print(f"\n{'='*70}")
print("PRACTICAL COMPLETE")
print("="*70)
print(f"\nKey Results:")
print(f"  • System successfully simulated for {t_span[1]} seconds")
print(f"  • Four numerical methods compared")
print(f"  • Energy conservation verified")
print(f"  • RK45 provides best balance of speed and accuracy")
print(f"  • Complete visualization produced")
print(f"\nLearning Outcomes Achieved:")
print(f"  ✓ Multi-body dynamics formulation")
print(f"  ✓ ODE solver implementation and comparison")
print(f"  ✓ Energy analysis and validation")
print(f"  ✓ Professional visualization")
print("="*70)

## Summary and Key Takeaways

### System Analysis

**Cart-Pendulum Dynamics**:
- Coupled 2-DOF system with motor forcing
- Lagrangian formulation yields coupled ODEs
- Energy flows between cart KE, pendulum KE, and PE
- Motor work input tracked and validated

### Numerical Methods

**Performance Rankings**:

1. **RK45** (Winner for most applications)
   - Good accuracy with reasonable cost
   - Adaptive step size
   - ~1000-2000 function evaluations
   - Excellent energy conservation

2. **DOP853** (High accuracy)
   - Best accuracy
   - Higher computational cost
   - ~2000-4000 function evaluations
   - Use when precision is critical

3. **RK23** (Quick solutions)
   - Lower accuracy
   - Fastest execution
   - ~500-1000 function evaluations
   - Good for rough estimates

4. **RK4 Manual** (Educational/special cases)
   - Fixed step size (predictable cost)
   - Good accuracy with small h
   - 4× steps function evaluations
   - Useful when adaptive stepping unwanted

### Validation Techniques

1. **Energy Conservation**
   - Track $E = KE_{cart} + KE_{pendulum} + PE$
   - Should equal $E_0 + W_{motor}$
   - Errors indicate numerical drift

2. **Method Comparison**
   - Use high-accuracy method as reference
   - Compare positions/velocities
   - Quantify accuracy vs cost trade-off

3. **Physical Reasonableness**
   - Visualize motion
   - Check for unphysical behavior
   - Verify constraint satisfaction

### Best Practices for Simulation Projects

1. **Formulation**
   - Use Lagrangian for complex systems
   - Identify all generalized coordinates
   - Convert to first-order ODEs carefully

2. **Implementation**
   - Write clean, modular ODE function
   - Test with simple cases first
   - Validate against known solutions

3. **Method Selection**
   - Start with RK45 (good default)
   - Use DOP853 for high accuracy
   - Consider stiff solvers if needed

4. **Validation**
   - Always check conservation laws
   - Compare methods
   - Visualize results
   - Verify physical correctness

5. **Documentation**
   - Clear variable names
   - Document equations
   - Explain method choices
   - Present results professionally

### Applications in Engineering

This cart-pendulum simulation approach applies to:

- **Robotics**: Manipulator dynamics, humanoid balance
- **Control Systems**: Inverted pendulum control, stabilization
- **Mechanical Design**: Crane systems, load handling
- **Automotive**: Suspension dynamics, rollover analysis
- **Aerospace**: Satellite attitude control, launch vehicle dynamics

### Course Completion

Congratulations! You've completed a comprehensive journey through:

1. **Kinematics**: Position, velocity, acceleration in multiple frames
2. **Lagrangian Mechanics**: Energy-based dynamics formulation
3. **Rigid Body Kinematics**: Rotation, angular velocity, Euler angles
4. **Particle/Rigid Body Kinetics**: Forces, torques, equations of motion
5. **Vibrations**: SDOF/MDOF systems, modal analysis
6. **Work & Energy**: Energy methods, momentum methods
7. **Numerical Simulation**: ODE solvers, multi-body dynamics

You now have the tools to:
- ✓ Formulate equations of motion for complex systems
- ✓ Simulate dynamic behavior numerically
- ✓ Analyze and validate results
- ✓ Apply to real engineering problems

**Next steps**: Apply these methods to your own projects, systems, and research!